# OneRPM Data Challenge

Este notebook executa automaticamente todo o pipeline do desafio

## Etapas

1. Verificar ambiente
2. Upload do projeto zipado
3. Descompactar o projeto
4. Instalar dependências
5. Executar a ingestão
6. Executar o dbt
7. Download do Arquivo Parquet

In [ ]:
import sys
import os

print("=" * 60)
print("Python:", sys.version)
print("Diretório atual:", os.getcwd())
print("=" * 60)

In [ ]:
from google.colab import files

print("Selecione o arquivo onerpm-data-challenge-colab.zip")

uploaded = files.upload()

In [ ]:
import zipfile

zip_name = next(iter(uploaded))

with zipfile.ZipFile(zip_name, "r") as zip_ref:
    zip_ref.extractall()

print("Projeto extraído com sucesso.")

In [ ]:
import os

os.chdir("onerpm-data-challenge-colab")

print("Diretório atual:")
print(os.getcwd())

In [ ]:
import subprocess
import sys

print("Instalando dependências...")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-r",
        "requirements.txt"
    ],
    check=True
)

print("Dependências instaladas.")

In [ ]:
import subprocess
import sys

print("=" * 60)
print("Executando ingestão...")
print("=" * 60)

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "ingestion.main"
    ],
    capture_output=True,
    text=True
)

print("STDOUT:")
print(result.stdout)

print("STDERR:")
print(result.stderr)

print("Return code:", result.returncode)

if result.returncode != 0:
    raise Exception("Erro na ingestão.")
else:
    print("Ingestão concluída com sucesso.")

In [ ]:
import subprocess

print("=" * 60)
print("Executando dbt...")
print("=" * 60)

process = subprocess.Popen(
    [
        "dbt",
        "run",
        "--profiles-dir",
        "."
    ],
    cwd="dbt",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for line in process.stdout:
    print(line, end="")

process.wait()

if process.returncode != 0:
    raise Exception("Erro na execução do dbt.")

print("\ndbt concluído com sucesso.")

In [ ]:
import duckdb
from pathlib import Path
from google.colab import files

print("=" * 60)
print("Exportando MART para Parquet...")
print("=" * 60)

db_path = Path("warehouse") / "onerpm.duckdb"

con = duckdb.connect(str(db_path))

output_path = "/content/mart_sales.parquet"

con.execute(f"""
COPY (

    SELECT *

    FROM mart_sales

)

TO '{output_path}'

(
    FORMAT PARQUET,
    COMPRESSION ZSTD
)
""")

con.close()

print(f"Arquivo gerado com sucesso:\n{output_path}")

files.download(output_path)

print("Download do Arquivo mart_sales.parquet gerado com sucesso!")

# Processo concluído

O pipeline foi executado com sucesso.

O arquivo `mart_sales.parquet` foi gerado e está pronto para ser utilizado no Power BI Desktop.